In [ ]:
%%bash
set -e

apt-get update -y
apt-get install -y xvfb x11-utils mesa-utils libgl1-mesa-glx libglib2.0-0
pip -q install ai2thor tqdm

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 6,555 B in 2s (4,153 B/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
libgl1-mesa-glx is

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
import os

os.makedirs("/content/pilot/scripts", exist_ok=True)
os.makedirs("/content/pilot/data", exist_ok=True)

In [ ]:
config_text = r'''
import os

# pilot/ 目录
PILOT_DIR = os.path.dirname(os.path.abspath(__file__))

# 数据目录
DATA_DIR = os.path.join(PILOT_DIR, "data")
STEP1_DIR = os.path.join(DATA_DIR, "step1_ground_truth")
STEP2_DIR = os.path.join(DATA_DIR, "step2_relations")
STEP3_DIR = os.path.join(DATA_DIR, "step3_text")

# 小规模 pilot：先只跑两个场景
SCENES = ["FloorPlan1", "FloorPlan2"]

# 每个场景最多保留多少对象
MAX_OBJECTS_PER_SCENE = 12

# 是否只保留 pickupable 物体
USE_PICKUPABLE_ONLY = False

# 随机种子
RANDOM_SEED = 42
'''

with open("/content/pilot/config.py", "w", encoding="utf-8") as f:
    f.write(config_text)

In [11]:
script_text = r'''
import json
import os
import random
import sys
from typing import Any, Dict, List, Set

from tqdm import tqdm
from ai2thor.controller import Controller


# 让脚本能找到 pilot/config.py
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PILOT_DIR = os.path.dirname(CURRENT_DIR)
PROJECT_ROOT = os.path.dirname(PILOT_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from pilot.config import (  # noqa: E402
    STEP1_DIR,
    SCENES,
    MAX_OBJECTS_PER_SCENE,
    USE_PICKUPABLE_ONLY,
    RANDOM_SEED,
)


random.seed(RANDOM_SEED)


def ensure_dir(path: str) -> None:
    """如果目录不存在则创建。"""
    os.makedirs(path, exist_ok=True)


def save_json(path: str, data: Any) -> None:
    """保存 JSON 文件。"""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def get_position(obj: Dict[str, Any]) -> Dict[str, float]:
    """从 AI2-THOR object metadata 中提取 position。"""
    pos = obj.get("position", {})
    return {
        "x": float(pos.get("x", 0.0)),
        "y": float(pos.get("y", 0.0)),
        "z": float(pos.get("z", 0.0)),
    }


def is_valid_object(obj: Dict[str, Any]) -> bool:
    """
    过滤不适合当前 pilot 的对象。
    这里保持规则尽量简单，方便后续排查和修改。
    """
    if "objectType" not in obj or "position" not in obj:
        return False

    obj_type = obj.get("objectType", "")

    # 先排除明显不需要的环境大类
    if obj_type in {"Floor", "Wall", "Room", "Ceiling"}:
        return False

    if USE_PICKUPABLE_ONLY and not obj.get("pickupable", False):
        return False

    return True


def build_object_record(obj: Dict[str, Any]) -> Dict[str, Any]:
    """
    构造 Step1 输出中的单个 object record。
    这里只保留后续 Step2/Step3 真正会用到的字段。
    """
    return {
        "objectId": obj.get("objectId"),
        "objectType": obj.get("objectType"),
        "position": get_position(obj),
        "pickupable": bool(obj.get("pickupable", False)),
        "receptacle": bool(obj.get("receptacle", False)),
        "openable": bool(obj.get("openable", False)),
        "visible": bool(obj.get("visible", False)),
        "parentReceptacles": obj.get("parentReceptacles") or [],
        "receptacleObjectIds": obj.get("receptacleObjectIds") or [],
    }


def build_object_index(objects: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """建立 objectId -> object metadata 的索引。"""
    return {
        obj["objectId"]: obj
        for obj in objects
        if obj.get("objectId") is not None
    }


def add_if_valid(
    obj: Dict[str, Any],
    selected: List[Dict[str, Any]],
    selected_ids: Set[str]
) -> bool:
    """
    如果对象合法且未被选中，则加入 selected。
    返回是否成功加入。
    """
    obj_id = obj.get("objectId")
    if obj_id is None:
        return False
    if obj_id in selected_ids:
        return False
    if not is_valid_object(obj):
        return False
    if len(selected) >= MAX_OBJECTS_PER_SCENE:
        return False

    selected.append(obj)
    selected_ids.add(obj_id)
    return True


def select_objects(raw_objects: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    关系友好型抽样：
    1) 先过滤合法对象
    2) 随机选 seed object
    3) 优先把 seed 的 parent receptacle 一起加入
    4) 如果 seed 本身是 receptacle，也可尝试加入其内部对象
    5) 最后若还没满，再随机补齐
    """
    filtered = [obj for obj in raw_objects if is_valid_object(obj)]
    object_index = build_object_index(raw_objects)

    random.shuffle(filtered)

    selected: List[Dict[str, Any]] = []
    selected_ids: Set[str] = set()

    for seed in filtered:
        if len(selected) >= MAX_OBJECTS_PER_SCENE:
            break

        # 先加 seed 自己
        added_seed = add_if_valid(seed, selected, selected_ids)
        if not added_seed:
            continue

        # 再尽量加入 seed 的父容器
        parents = seed.get("parentReceptacles") or []
        for parent_id in parents:
            if len(selected) >= MAX_OBJECTS_PER_SCENE:
                break
            parent_obj = object_index.get(parent_id)
            if parent_obj is not None:
                add_if_valid(parent_obj, selected, selected_ids)

        # 如果 seed 是 receptacle，尝试加入其内部对象（少量）
        if seed.get("receptacle", False):
            child_ids = seed.get("receptacleObjectIds") or []
            random.shuffle(child_ids)
            for child_id in child_ids[:2]:
                if len(selected) >= MAX_OBJECTS_PER_SCENE:
                    break
                child_obj = object_index.get(child_id)
                if child_obj is not None:
                    add_if_valid(child_obj, selected, selected_ids)

    # 如果还没满，再随机补齐
    if len(selected) < MAX_OBJECTS_PER_SCENE:
        leftovers = [obj for obj in filtered if obj.get("objectId") not in selected_ids]
        random.shuffle(leftovers)
        for obj in leftovers:
            if len(selected) >= MAX_OBJECTS_PER_SCENE:
                break
            add_if_valid(obj, selected, selected_ids)

    return selected


def extract_scene_ground_truth(controller: Controller, scene: str) -> Dict[str, Any]:
    """
    对单个 scene 执行 Step1：
    reset scene -> 读取 metadata -> 保留少量对象 -> 生成 JSON 结构
    """
    event = controller.reset(scene=scene)
    metadata = event.metadata

    raw_objects = metadata.get("objects", [])
    selected_objects = select_objects(raw_objects)
    object_records = [build_object_record(obj) for obj in selected_objects]

    return {
        "scene": scene,
        "num_all_objects": len(raw_objects),
        "num_selected_objects": len(object_records),
        "objects": object_records,
    }


def main() -> None:
    ensure_dir(STEP1_DIR)

    print("Starting Step1: Spatial Ground Truth Construction", flush=True)
    print(f"Scenes: {SCENES}", flush=True)
    print(f"Output dir: {STEP1_DIR}", flush=True)
    print(f"Max objects per scene: {MAX_OBJECTS_PER_SCENE}", flush=True)
    print(f"USE_PICKUPABLE_ONLY: {USE_PICKUPABLE_ONLY}", flush=True)

    controller = None
    try:
        controller = Controller(
            scene=SCENES[0],
            width=300,
            height=300,
            agentMode="default",
        )

        for scene in tqdm(SCENES, desc="Extracting scenes"):
            scene_data = extract_scene_ground_truth(controller, scene)
            out_path = os.path.join(STEP1_DIR, f"{scene}.json")
            save_json(out_path, scene_data)
            print(
                f"Saved {scene}: "
                f"{scene_data['num_selected_objects']} / {scene_data['num_all_objects']} objects",
                flush=True,
            )

        print("Step1 completed.", flush=True)

    finally:
        if controller is not None:
            controller.stop()


if __name__ == "__main__":
    main()
'''

with open("/content/pilot/scripts/step1_extract_ground_truth.py", "w", encoding="utf-8") as f:
    f.write(script_text)

print("Updated step1_extract_ground_truth.py written successfully.")

Updated step1_extract_ground_truth.py written successfully.


In [12]:
%%bash
set -e

xvfb-run -s "-screen 0 1024x768x24" \
env LIBGL_ALWAYS_SOFTWARE=1 \
python /content/pilot/scripts/step1_extract_ground_truth.py

Starting Step1: Spatial Ground Truth Construction
Scenes: ['FloorPlan1', 'FloorPlan2']
Output dir: /content/pilot/data/step1_ground_truth
Max objects per scene: 12
USE_PICKUPABLE_ONLY: False
Extracting scenes: 100%|██████████| 2/2 [00:23<00:00, 11.68s/it]
Step1 completed.


In [13]:
!find /content/pilot/data/step1_ground_truth -maxdepth 2 -type f

/content/pilot/data/step1_ground_truth/FloorPlan2.json
/content/pilot/data/step1_ground_truth/FloorPlan1.json


In [14]:
!zip -r /content/step1_ground_truth.zip /content/pilot/data/step1_ground_truth

updating: content/pilot/data/step1_ground_truth/ (stored 0%)
updating: content/pilot/data/step1_ground_truth/FloorPlan2.json (deflated 81%)
updating: content/pilot/data/step1_ground_truth/FloorPlan1.json (deflated 80%)


In [ ]:
from google.colab import files
files.download('/content/step1_ground_truth.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import json
from pprint import pprint

path = "/content/pilot/data/step1_ground_truth/FloorPlan1.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=== Top level keys ===")
print(data.keys())

print("\nScene:", data["scene"])
print("Num all objects:", data["num_all_objects"])
print("Num selected objects:", data["num_selected_objects"])

print("\n=== One object example ===")
pprint(data["objects"][0])

=== Top level keys ===
dict_keys(['scene', 'num_all_objects', 'num_selected_objects', 'objects'])

Scene: FloorPlan1
Num all objects: 77
Num selected objects: 12

=== One object example ===
{'objectId': 'Fork|+00.95|+00.77|-02.37',
 'objectType': 'Fork',
 'openable': False,
 'parentReceptacles': ['Drawer|+00.95|+00.83|-02.20'],
 'pickupable': True,
 'position': {'x': 0.9492349624633789,
              'y': 0.7680964469909668,
              'z': -2.3652503490448},
 'receptacle': False,
 'receptacleObjectIds': [],
 'visible': False}


In [16]:
import json

with open('/content/pilot/data/step1_ground_truth/FloorPlan1.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

for i, obj in enumerate(data["objects"]):
    print(i, obj["objectType"], "| receptacle =", obj["receptacle"], "| parents =", obj["parentReceptacles"])

0 Fork | receptacle = False | parents = ['Drawer|+00.95|+00.83|-02.20']
1 Drawer | receptacle = True | parents = []
2 Pot | receptacle = True | parents = ['CounterTop|-01.87|+00.95|-01.21']
3 CounterTop | receptacle = True | parents = []
4 Cabinet | receptacle = True | parents = []
5 Drawer | receptacle = True | parents = []
6 Knife | receptacle = False | parents = ['Drawer|-01.56|+00.84|-00.20']
7 Plate | receptacle = True | parents = ['Cabinet|+00.72|+02.02|-02.46']
8 Cabinet | receptacle = True | parents = []
9 HousePlant | receptacle = False | parents = ['CounterTop|-01.87|+00.95|-01.21']
10 SaltShaker | receptacle = False | parents = ['CounterTop|+00.69|+00.95|-02.48']
11 CounterTop | receptacle = True | parents = []


In [17]:
import json

with open('/content/pilot/data/step1_ground_truth/FloorPlan1.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

selected_ids = {obj["objectId"] for obj in data["objects"]}

print("=== parent membership check ===")
for obj in data["objects"]:
    for p in obj["parentReceptacles"]:
        print(
            f'{obj["objectType"]} | parent = {p} | in_selected = {p in selected_ids}'
        )

=== parent membership check ===
Fork | parent = Drawer|+00.95|+00.83|-02.20 | in_selected = True
Pot | parent = CounterTop|-01.87|+00.95|-01.21 | in_selected = True
Knife | parent = Drawer|-01.56|+00.84|-00.20 | in_selected = True
Plate | parent = Cabinet|+00.72|+02.02|-02.46 | in_selected = True
HousePlant | parent = CounterTop|-01.87|+00.95|-01.21 | in_selected = True
SaltShaker | parent = CounterTop|+00.69|+00.95|-02.48 | in_selected = True
